In [1]:
# Cài đặt model
#pip install statsmodels
#pip install openpyxl
#!pip install seaborn
#!pip install networkx

## 1. Khởi tạo môi trường & import thư viện

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import jarque_bera, shapiro
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import acf, pacf
import statsmodels.api as sm

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

def hline(text=""):
    print("-" * 80)
    if text:
        print("📌 KẾT LUẬN / DIỄN GIẢI:")
        print(text)
    print("-" * 80)

## 2. Nạp dữ liệu gốc & Tiền xử lý dữ liệu ( Xoá dòng trùng )

In [2]:
import pandas as pd
import numpy as np
from scipy.io import savemat


# 1. Đọc dữ liệu gốc
file_path = "C:/Users/Quoc Dat/Downloads/FINAL DATASET.xlsx"
df_raw = pd.read_excel(file_path)

# 2. Chuẩn hoá tên cột
df = df_raw.copy()
df.columns = [c.lower() for c in df.columns]

# 3. Chuyển time sang datetime
df["time"] = pd.to_datetime(df["time"])

# 4. Lọc giai đoạn 24/03/2021 -> 10/11/2025
start_date = "2021-03-24"
end_date   = "2025-11-10"
df = df[(df["time"] >= start_date) & (df["time"] <= end_date)].copy()

# 5. Loại bỏ mã ROS
df = df[df["symbol"] != "ROS"].copy()

# 6. Loại bỏ dòng trùng (time, symbol)
before = df.shape[0]
df = df.drop_duplicates(subset=["time", "symbol"], keep="first")
after = df.shape[0]

print("Kích thước sau lọc & xoá trùng:", df.shape)
print("Số dòng trùng đã loại:", before - after)
print("Số mã cổ phiếu:", df["symbol"].nunique())
print("Khoảng thời gian:", df["time"].min(), "→", df["time"].max())

Kích thước sau lọc & xoá trùng: (47475, 9)
Số dòng trùng đã loại: 11224
Số mã cổ phiếu: 41
Khoảng thời gian: 2021-03-24 00:00:00 → 2025-11-10 00:00:00


In [3]:
# 1. Pivot: time × symbol → close price
prices = df.pivot(index="time", columns="symbol", values="close")

# 2. Tính log-return
log_returns = np.log(prices).diff()
log_returns = log_returns.dropna(how="all")

print("prices shape:", prices.shape)
print("log_returns shape:", log_returns.shape)
print("Số mã trong log_returns:", len(log_returns.columns))
print("Một vài mã:", list(log_returns.columns)[:10])

prices shape: (1158, 41)
log_returns shape: (1157, 41)
Số mã trong log_returns: 41
Một vài mã: ['ACB', 'BCM', 'BID', 'BVH', 'CTG', 'DGC', 'EIB', 'FPT', 'GAS', 'GVR']


In [21]:
exclude = ["VNINDEX", "VN30", "VN30INDEX"]
df_stock = df[~df["symbol"].isin(exclude)].copy()

# 2. Tính thanh khoản trung bình theo mã (mean volume)
liq_stats = (
    df_stock.groupby("symbol")["volume"]
            .mean()
            .sort_values(ascending=False)
)

print("Top 30 mã theo volume trung bình:")
print(liq_stats.head(30))

# 3. Lấy top 20 mã thanh khoản cao nhất
top20_symbols = list(liq_stats.head(20).index)
print("\n20 mã được chọn:", top20_symbols)

top20_symbols = [s for s in top20_symbols if s in log_returns.columns]
logret_sel = log_returns[top20_symbols].copy()

print("Số mã dùng cho PMIME:", len(top20_symbols))
print("logret_sel shape:", logret_sel.shape)

Top 30 mã theo volume trung bình:
symbol
SHB    2.751120e+07
HPG    2.745566e+07
SSI    2.022881e+07
VPB    1.890031e+07
STB    1.786747e+07
MBB    1.580805e+07
NVL    1.512040e+07
TCB    1.100947e+07
TPB    9.363937e+06
PDR    8.750126e+06
TCH    8.073443e+06
CTG    7.995802e+06
ACB    7.846191e+06
HDB    6.759227e+06
LPB    6.711772e+06
VRE    6.337276e+06
VHM    6.083625e+06
EIB    5.986979e+06
MWG    5.294228e+06
VIB    5.184610e+06
VIC    3.445698e+06
FPT    3.428110e+06
GVR    3.300398e+06
MSN    3.271430e+06
VNM    3.202923e+06
SBT    2.866761e+06
KDH    2.728264e+06
BID    2.559828e+06
DGC    2.255333e+06
VCB    1.982209e+06
Name: volume, dtype: float64

20 mã được chọn: ['SHB', 'HPG', 'SSI', 'VPB', 'STB', 'MBB', 'NVL', 'TCB', 'TPB', 'PDR', 'TCH', 'CTG', 'ACB', 'HDB', 'LPB', 'VRE', 'VHM', 'EIB', 'MWG', 'VIB']
Số mã dùng cho PMIME: 20
logret_sel shape: (1157, 20)


# 3. Top 20 cổ phiếu có biến động lớn nhất

In [4]:
import pandas as pd
import numpy as np
from scipy.io import savemat  

# 1. Đọc dữ liệu
file_path = "C:/Users/Quoc Dat/Downloads/FINAL DATASET.xlsx"
df_raw = pd.read_excel(file_path)

# 2. Chuẩn hoá cột
df = df_raw.copy()
df.columns = [c.lower() for c in df.columns]

# 3. Chuyển time sang datetime
df["time"] = pd.to_datetime(df["time"])

# 4. Lọc khoảng thời gian
start_date = "2021-03-24"
end_date   = "2025-11-10"
df = df[(df["time"] >= start_date) & (df["time"] <= end_date)].copy()

# 5. Loại ROS
df = df[df["symbol"] != "ROS"].copy()

# 6. Loại trùng
df = df.drop_duplicates(subset=["time", "symbol"], keep="first")

print(df.shape)

(47475, 9)


In [5]:
# Pivot: time × symbol → close price
prices = df.pivot(index="time", columns="symbol", values="close")

# Tính log-return
log_returns = np.log(prices).diff().dropna(how="all")

print("log_returns shape:", log_returns.shape)
print("Một vài mã:", list(log_returns.columns)[:10])

log_returns shape: (1157, 41)
Một vài mã: ['ACB', 'BCM', 'BID', 'BVH', 'CTG', 'DGC', 'EIB', 'FPT', 'GAS', 'GVR']


In [6]:
df = df[df["volume"] > 0].copy()              # tránh lỗi log(0)
df["log_volume"] = np.log(df["volume"])

# Sai phân log-volume theo từng mã
df = df.sort_values(["symbol", "time"])
df["log_vol_return"] = df.groupby("symbol")["log_volume"].diff()

In [7]:
EXCLUDE = ["VNINDEX", "VN30", "VN30INDEX", "ROS"]

# Lọc mã cổ phiếu
stock_symbols = [s for s in log_returns.columns if s not in EXCLUDE]

# Volatility = độ lệch chuẩn log-return
vol_stats = (
    log_returns[stock_symbols]
      .std()
      .sort_values(ascending=False)
)

print("\nTop 20 volatility:")
print(vol_stats.head(20))

# Danh sách 20 mã
top20_volatile_symbols = list(vol_stats.head(20).index)
print(top20_volatile_symbols)

# Log-return của 20 mã
logret_volatile = log_returns[top20_volatile_symbols].copy()


Top 20 volatility:
symbol
TCH    0.029711
GVR    0.028046
PDR    0.027269
SSI    0.026762
NVL    0.026630
DGC    0.025876
EIB    0.025280
LPB    0.024593
SHB    0.024236
STB    0.024117
VRE    0.023258
SBT    0.022973
MSN    0.022575
KDH    0.022186
MWG    0.022116
BCM    0.021617
HPG    0.021542
TPB    0.021524
VIC    0.021380
VHM    0.021011
dtype: float64
['TCH', 'GVR', 'PDR', 'SSI', 'NVL', 'DGC', 'EIB', 'LPB', 'SHB', 'STB', 'VRE', 'SBT', 'MSN', 'KDH', 'MWG', 'BCM', 'HPG', 'TPB', 'VIC', 'VHM']


In [8]:
DATE_COL   = "time"
SYMBOL_COL = "symbol"

# usdreturn
usd_series = (
    df.groupby(DATE_COL)["usdreturn"]
      .first()
      .reindex(logret_volatile.index)
)

# order_avg
order_series = (
    df[df[SYMBOL_COL].isin(top20_volatile_symbols)]
      .groupby(DATE_COL)["order"]
      .mean()
      .reindex(logret_volatile.index)
)

# volavg = trung bình log-vol-return của 20 mã
vol_series = (
    df[df[SYMBOL_COL].isin(top20_volatile_symbols)]
      .groupby(DATE_COL)["log_vol_return"]
      .mean()
      .reindex(logret_volatile.index)
)

In [9]:
data_volatile = logret_volatile.copy()
data_volatile["usdreturn"] = usd_series
data_volatile["order_avg"] = order_series
data_volatile["volavg"]    = vol_series

# Loại mọi dòng thiếu dữ liệu
data_volatile = data_volatile.dropna(how="any")

print("Final shape:", data_volatile.shape)
print("Biến gồm:", list(data_volatile.columns))

Final shape: (1153, 23)
Biến gồm: ['TCH', 'GVR', 'PDR', 'SSI', 'NVL', 'DGC', 'EIB', 'LPB', 'SHB', 'STB', 'VRE', 'SBT', 'MSN', 'KDH', 'MWG', 'BCM', 'HPG', 'TPB', 'VIC', 'VHM', 'usdreturn', 'order_avg', 'volavg']


In [ ]:
# Tổng số ô bị thiếu
total_na = data_top20_volume.isna().sum().sum()
print("Tổng số giá trị NA:", total_na)

# Số NA theo từng biến
print("\nSố NA theo từng biến:")
print(data_top20_volume.isna().sum())

# Số dòng có ít nhất 1 NA
rows_with_na = data_top20_volume.isna().any(axis=1).sum()
print("\nSố dòng có ít nhất 1 NA:", rows_with_na)

In [10]:
X = data_volatile.values
var_names = np.array(data_volatile.columns, dtype=object)

savemat(
    "pmime_input_top20volatile_23vars.mat",
    {"X": X, "var_names": var_names}
)

print("Đã lưu file pmime_input_top20volatile_23vars.mat")

Đã lưu file pmime_input_top20volatile_23vars.mat


# 4. Top 20 mã có Khối lượng cao nhất

In [16]:
import pandas as pd
import numpy as np
from scipy.io import savemat  

# 1. Đọc dữ liệu
file_path = "C:/Users/Quoc Dat/Downloads/FINAL DATASET.xlsx"
df_raw = pd.read_excel(file_path)

# 2. Chuẩn hoá cột
df = df_raw.copy()
df.columns = [c.lower() for c in df.columns]

# 3. Chuyển time sang datetime
df["time"] = pd.to_datetime(df["time"])

# 4. Lọc khoảng thời gian
start_date = "2021-03-24"
end_date   = "2025-11-10"
df = df[(df["time"] >= start_date) & (df["time"] <= end_date)].copy()

# 5. Loại ROS
df = df[df["symbol"] != "ROS"].copy()

# 6. Loại trùng
df = df.drop_duplicates(subset=["time", "symbol"], keep="first")

print(df.shape)

(47475, 9)


In [17]:
prices = df.pivot(index="time", columns="symbol", values="close")
log_returns = np.log(prices).diff()
log_returns = log_returns.dropna(how="all")

In [18]:
EXCLUDE = ["VNINDEX", "VN30", "VN30INDEX", "ROS"]

# Thanh khoản trung bình mỗi mã (loại index / ROS)
df_liq = df[~df["symbol"].isin(EXCLUDE)].copy()

liq_stats = (
    df_liq
      .groupby("symbol")["volume"]
      .mean()
      .sort_values(ascending=False)
)

print("\nTop 20 mã theo thanh khoản (volume trung bình):")
print(liq_stats.head(20))

top20_volume_symbols = liq_stats.head(20).index.tolist()
print("\n20 mã volume cao nhất:", top20_volume_symbols)

# Log-return của 20 mã này
logret_top20_vol = log_returns[top20_volume_symbols].copy()
print("logret_top20_vol shape:", logret_top20_vol.shape)


Top 20 mã theo thanh khoản (volume trung bình):
symbol
SHB    2.751120e+07
HPG    2.745566e+07
SSI    2.022881e+07
VPB    1.890031e+07
STB    1.786747e+07
MBB    1.580805e+07
NVL    1.512040e+07
TCB    1.100947e+07
TPB    9.363937e+06
PDR    8.750126e+06
TCH    8.073443e+06
CTG    7.995802e+06
ACB    7.846191e+06
HDB    6.759227e+06
LPB    6.711772e+06
VRE    6.337276e+06
VHM    6.083625e+06
EIB    5.986979e+06
MWG    5.294228e+06
VIB    5.184610e+06
Name: volume, dtype: float64

20 mã volume cao nhất: ['SHB', 'HPG', 'SSI', 'VPB', 'STB', 'MBB', 'NVL', 'TCB', 'TPB', 'PDR', 'TCH', 'CTG', 'ACB', 'HDB', 'LPB', 'VRE', 'VHM', 'EIB', 'MWG', 'VIB']
logret_top20_vol shape: (1157, 20)


In [19]:
# Đảm bảo không có volume <= 0
df = df[df["volume"] > 0].copy()

# Log(volume)
df["log_volume"] = np.log(df["volume"])

# Sai phân log-volume theo từng mã (thay đổi thanh khoản theo thời gian)
df = df.sort_values(["symbol", "time"])
df["log_vol_return"] = df.groupby("symbol")["log_volume"].diff()

In [20]:
DATE_COL   = "time"
SYMBOL_COL = "symbol"

# 1) usdreturn mỗi ngày (chung toàn thị trường)
usd_series = (
    df.groupby(DATE_COL)["usdreturn"]
      .first()
      .reindex(logret_top20_vol.index)
)

# 2) order_avg = trung bình order của 20 mã volume cao nhất trong ngày
order_series = (
    df[df[SYMBOL_COL].isin(top20_volume_symbols)]
      .groupby(DATE_COL)["order"]
      .mean()
      .reindex(logret_top20_vol.index)
)

# 3) volavg = trung bình log_vol_return của 20 mã volume cao nhất trong ngày
vol_series = (
    df[df[SYMBOL_COL].isin(top20_volume_symbols)]
      .groupby(DATE_COL)["log_vol_return"]
      .mean()
      .reindex(logret_top20_vol.index)
)

# Dataset cuối cho PMIME
data_top20_volume = logret_top20_vol.copy()
data_top20_volume["usdreturn"] = usd_series
data_top20_volume["order_avg"] = order_series
data_top20_volume["volavg"]    = vol_series

data_top20_volume = data_top20_volume.dropna(how="any")

print("Final data_top20_volume shape:", data_top20_volume.shape)
print("Biến:", list(data_top20_volume.columns))

Final data_top20_volume shape: (1153, 23)
Biến: ['SHB', 'HPG', 'SSI', 'VPB', 'STB', 'MBB', 'NVL', 'TCB', 'TPB', 'PDR', 'TCH', 'CTG', 'ACB', 'HDB', 'LPB', 'VRE', 'VHM', 'EIB', 'MWG', 'VIB', 'usdreturn', 'order_avg', 'volavg']


In [22]:
# Tổng số ô bị thiếu
total_na = data_top20_volume.isna().sum().sum()
print("Tổng số giá trị NA:", total_na)

# Số NA theo từng biến
print("\nSố NA theo từng biến:")
print(data_top20_volume.isna().sum())

# Số dòng có ít nhất 1 NA
rows_with_na = data_top20_volume.isna().any(axis=1).sum()
print("\nSố dòng có ít nhất 1 NA:", rows_with_na)

Tổng số giá trị NA: 0

Số NA theo từng biến:
symbol
SHB          0
HPG          0
SSI          0
VPB          0
STB          0
MBB          0
NVL          0
TCB          0
TPB          0
PDR          0
TCH          0
CTG          0
ACB          0
HDB          0
LPB          0
VRE          0
VHM          0
EIB          0
MWG          0
VIB          0
usdreturn    0
order_avg    0
volavg       0
dtype: int64

Số dòng có ít nhất 1 NA: 0


In [25]:
from scipy.io import savemat
import numpy as np

# Ma trận X: (T × K), K = 23 biến
X = data_top20_volume.values

# Tên biến
var_names = np.array(data_top20_volume.columns, dtype=object)

print("Kích thước X:", X.shape)
print("Số biến:", len(var_names))
print("Tên biến:", var_names)

# Xuất file .mat
savemat(
    "pmime_input_top20volume_23vars.mat",
    {"X": X, "var_names": var_names}
)

print("ĐÃ LƯU: pmime_input_top20volume_23vars.mat")

Kích thước X: (1153, 23)
Số biến: 23
Tên biến: ['SHB' 'HPG' 'SSI' 'VPB' 'STB' 'MBB' 'NVL' 'TCB' 'TPB' 'PDR' 'TCH' 'CTG'
 'ACB' 'HDB' 'LPB' 'VRE' 'VHM' 'EIB' 'MWG' 'VIB' 'usdreturn' 'order_avg'
 'volavg']
ĐÃ LƯU: pmime_input_top20volume_23vars.mat
